In [6]:
from abc import ABC, abstractclassmethod, abstractproperty
from datetime import datetime
import textwrap #bonus na aula de resolucao do problema

##parte 01, fazendo a orientação a objetos
#classe cliente recebe como argumento o endereco
class Cliente:
     def __init__(self, endereco):
         self.endereco = endereco
         self.contas = [] #iniciando as contas comom vazio

     def realizar_transacao(self, conta, transacao):
         transacao.registrar(conta)

     def adicionar_conta(self, conta):
         self.contas.append(conta)#adiciona a conta recebida por padrao, dentro do array de contas

#Herda da classe pai CLIENTE
class PessoaFisica(Cliente):
     def __init__(self, nome, data_nascimento, cpf, endereco):
         super().__init__(endereco)#chamando o construtor a classe pai
         self.nome = nome
         self.data_nascimento = data_nascimento
         self.cpf = cpf

#Classe Conta recebe argumentos numero
class Conta:
     def __init__(self, numero, cliente):
         self._saldo = 0
         self._numero = numero
         self._agencia = "0001"
         self._cliente = cliente
         self._historico = Historico()

    #mapeando os classmethods
     @classmethod
     def nova_conta(cls, cliente, numero):
         return cls(numero, cliente)
     
     @property
     def saldo(self):
         return self._saldo
     
     @property
     def numero(self):
         return self._numero
     
     @property
     def agencia(self):
         return self._agencia
     
     @property
     def cliente(self):
         return self._cliente
     
     @property
     def historico(self):
         return self._historico
     
    #definindo o metodo sacar
     def sacar(self, valor):
         saldo = self.saldo
         excedeu_saldo = valor > saldo

         if excedeu_saldo:
             print("\n@@@ Operação inválida! Saldo insuficiente. @@@")
            
         elif valor > 0:
             self._saldo -= valor
             print("\n== Saque realizado com sucesso! ==")
             return True
         
         else:
             print('\n@@@ Operação inválida! O valor informado deve ser positivo. @@@')
             return False
         
    #definindo o metodo depositar
     def depositar(self, valor):
         if valor >0:
             self._saldo += valor
             print('\n== Depósito realizado com sucesso! ==')
         else:
             print('\n@@@ Operação inválida! O valor informado deve ser positivo. @@@')
             return False
         
         return True

#Herda da classe pai CONTA
class ContaCorrente(Conta):
     def __init__(self, numero, cliente, limite=500, limite_saques=3):
         super().__init__(numero, cliente)
         self.limite = limite
         self.limite_saques = limite_saques

    #difinindo o metodo sacar
     def sacar(self, valor):
         numero_saques = len(
             [transacao for transacao in self.historico.transacoes if transacao["tipo"] == Saque.__name__]#fazendo a leitura dos saques com a selecao do tipo do historico.
         )

         excedeu_limite = valor > self.limite
         excedeu_saques = numero_saques >= self.limite_saques

         if excedeu_limite:
             print("\n@@@ Operação falhou! O valor solicitado excede o seu limite. @@@")
         elif excedeu_saques:
             print("\n@@@ Operação falhou! Quantidade máxima de saques excedida. @@@")
         else:
             return super().sacar(valor)
         return False
     
     #representação da classe
     def __str__(self):
         return f"""\
         Agência: {self.agencia}
         C/C: {self.numero}
         Titular: {self.cliente.nome}
         """

#classe historico
class Historico:
     def __init__(self):
         self._transacoes = []

     @property
     def transacoes(self):
         return self.transacoes
     
     def adicionar_transacao(self, transacao):
         self._transacoes.append(
             {
                 "tipo": transacao.__class__.__name__,
                 "valor": transacao.valor,
                 "data": datetime.now().strftime("%d-%m-%Y %H:%M:%s"),
             }
         )

#classe transacao é abstrata, por isso estende o modulo ABC
class Transacao(ABC):
     @property
     @abstractproperty
     def valor(self):
         pass
     
     @abstractclassmethod
     def registrar(cls, conta):
         pass

#Herda da classe pai TRANSACAO
class Saque(Transacao):
     def __init__(self, valor):
         self._valor = valor

     @property
     def valor(self):
         return self._valor
     
     def registrar(self, conta):
         sucesso_transacao = conta.sacar(self.valor)

         if sucesso_transacao:
             conta.historico.adicionar_transacao(self)

#Herda da classe pai TRANSACAO
class Deposito(Transacao):
     def __init__(self, valor):
         self._valor = valor

     @property
     def valor(self):
         return self._valor
     
     def registrar(self, conta):
         sucesso_transacao = conta.depositar(self.valor)

         if sucesso_transacao:
             conta.historico.adicionar_transacao(self)


##parte 02, tudo funcioando com a orientacao a objetos
#declando em funções
def menu():
    menu = '''\n
    =============== MENU ===============
    [1]\tDepositar
    [2]\tSacar
    [3]\tExtrato
    [4]\tNova Conta
    [5]\tListar Contas
    [6]\tNovo Usuário
    [0]\tSair
    => '''
    return input(textwrap.dedent(menu))

#definindo metodo para recuperar a conta do cliente desejado OK
def recuperar_conta_cliente(cliente):
    if not cliente.contas:
        print('\n@@@ Cliente não possui conta em nosso banco! @@@')
        return
    
    #FIXME: não permite o cliente escolher a conta
    return cliente.contas[0]

#declarando função depositar OK
def depositar(clientes):
    cpf = input('Digite o CPF (somente números): ')
    cliente = filtrar_clientes(cpf, clientes)#filtrando o cpf do cliente

    if not cliente:
        print('\n@@@ Cliente não cadastrado(a)! @@@')
        return

    valor = float(input('Insira o valor do depósito: '))
    transacao = Deposito(valor)

    conta = recuperar_conta_cliente(cliente)
    if not conta:
        return
    
    cliente.realizar_transacao(conta, transacao)

#declarando função sacar OK
def sacar(clientes):
    cpf = input('Digite o CPF (somente números): ')
    cliente = filtrar_clientes(cpf, clientes)#filtrando o cpf do cliente
    
    if not cliente:
        print('\n@@@ Cliente não localizado. @@@')
        return
    
    valor = float(input('Insira o valor do saque: '))
    transacao = Saque(valor)

    conta = recuperar_conta_cliente(cliente)
    if not conta:
        return
    
    cliente.realizar_transacao(conta, transacao)

#declarando função extrato OK
def exibir_extrato(clientes):
    cpf = input('Digite o CPF (somente números): ')
    cliente = filtrar_clientes(cpf, clientes)#filtrando o cpf do cliente

    if not cliente:
        print('\n@@@ Cliente não localizado. @@@')
        return
    
    conta = recuperar_conta_cliente(cliente)
    if not conta:
        return
    
    print('\n========== Extrato ==========')
    transacoes = conta.historico.transacoes

    extrato = ""
    if not transacoes:
        extrato = 'Não foram realizadas movimentações.'
    else:
        for transacao in transacoes:
            extrato += f'\n{transacao['tipo']}:\n\tR${transacao['valor']:.2f}'

    print(extrato)
    print(f'\nSaldo:\n\tR$ {conta.saldo:.2f}')
    print('==========================================')

#criando e declarando funcao de novo usuario OK
def criar_cliente(clientes):
    cpf = input('Digite o CPF (somente números): ')
    usuario = filtrar_usuario(cpf, usuarios)
    
    if cliente:
        print('\n@@@ Usuário já cadastrado, informe um novo CPF. @@@')
        return
    
    nome = input('Digite o nome completo: ')
    data_nascimento = input('Digite a data de nascimento (dd-mm-aaaa): ')
    endereco = input('Digite o endereço (nome da rua, nº - bairro - cidade/sigla estado): ')

    cliente = PessoaFisica(nome=nome, data_nascimento=data_nascimento, cpf=cpf, endereco=endereco)

    clietes.append(cliente)

    print('=== Cliente cadastrado com sucesso! ===')

#funcao para filtrar cliente OK
def filtrar_cliente(cpf, clientes):
    clientes_filtrados = [cliente for cliente in
    clientes if cliente.cpf == cpf]
    return clientes_filtrados[0] if clientes_filtrados else None

#criando e declarando funcao de criar conta OK
def criar_conta(numero_conta, clientes, contas):
    cpf = input('Digite o CPF do cliente: ')
    usuario = filtrar_usuario(cpf, usuarios)

    if not cliente:
        print('\n@@@ Cliente não localizado, fluxo de criação de conta encerrado! @@@')
        return
        
    conta = ContaCorrente.nova_conta(cliente = cliente, numero = numero_conta)
    contas.append(conta)
    cliente.contas.append(conta)

    print('\n== Conta criada com sucesso! ==')

#funcao de listar contas OK
def listar_contas(contas):
    for conta in contas:
        print('=' * 100)
        print(textwrap.dedent(str(conta)))

#declarando toda estrutura do programa
def main():
    clientes = []
    contas = []

    while True:
        opcao = menu()
        if opcao == '1':
            depositar(clientes)

        elif opcao == '2':
            sacar(clientes)

        elif opcao == '3':
            exibir_extrato

        elif opcao == '4':
            numer_conta = len(contas) + 1
            criar_conta(numero_conta, clientes, contas)

        elif opcao == '5':
            listar_contas(contas)

        elif opcao == '6':
            criar_cliente(clientes)

        elif opcao == '0':
            break

        else:
            print('Opção inválida, digite uma valor que pertence ao menu.')#mensagem de erro caso o numero selecionado não pertença ao menu.

main()